# Task 1.3 — Baselines, Limitations, and Where MSVMpack Falls Short

## 1. Primary Baseline: BSVM (C++)
The primary baseline named in the MSVMpack paper is **BSVM** — a C++ implementation of the **Weston & Watkins (WW)** multi-class SVM by Hsu & Lin.
It is highly optimised for the WW formulation and represents the state-of-the-art single-model solver against which MSVMpack is benchmarked on both training speed (Table 2) and test error.

## 2. Limitations of the Prior Art (BSVM and contemporaries)
1. **No unified package:** Each of WW, CS, LLW, MSVM2 had its own separate, incompatible implementation — no single tool covered all four models under one interface.
2. **Memory failure on large datasets:** BSVM attempts to load the full $n\times n$ kernel matrix into RAM and **crashes with an out-of-memory error on CB513** (84,119 samples) — the matrix would require tens of gigabytes.
3. **No parallel support:** All prior single-model tools were single-threaded, with no multi-core training or cross-validation parallelism.

## 3. How MSVMpack Overcomes These Limitations
1. **Unified C implementation:** One shared library, one API — switch between WW/CS/LLW/MSVM2 by changing a single parameter.
2. **Decomposition method:** Frank-Wolfe with working-set LP keeps memory proportional to the working-set size, not $n^2$ — CB513 trains without memory errors.
3. **Parallel training:** OpenMP parallelism for both model fitting and cross-validated hyperparameter search, giving wall-clock speedups on multi-core machines.

## 4. Scenarios Where MSVMpack Does NOT Outperform the Baseline

**Small datasets — USPS-500, WW model (Table 2):**  
On a 500-sample subset of USPS with the WW model, BSVM trains in ~0.2 s while MSVMpack takes ~2.5 s.
The Frank-Wolfe working-set management and decomposition bookkeeping add fixed per-iteration overhead that outweighs any memory benefit when $n$ is small enough for BSVM to fit the full kernel matrix comfortably.

**Large number of classes with MSVM2, Q > 20 (Section 3.1):**  
Computing $U(\alpha)$ for MSVM2 requires solving an embedded QP sub-problem at every Frank-Wolfe iteration.
When $Q>20$ this embedded problem grows quadratically in $Q$, making the stopping-criterion evaluation the dominant runtime cost and erasing MSVMpack's advantage over simpler single-model solvers for scenarios with many classes.